# GenAI Model Access Layer and Structured Outputs

This course covers production-grade patterns for integrating LLM APIs, structuring and validating model outputs, advanced prompting, function calling, multimodal pipelines, fine-tuning, and ensuring output quality and reliability. Each module includes practical Python demos using the OpenAI API.

## Table of Contents

1. [Module 01: Production-ready API Patterns for GenAI](#module-01)
2. [Module 02: Structuring and Validating Output Generation](#module-02)
3. [Module 03: Advanced Prompting for Production Systems](#module-03)
4. [Module 04: Function Calling Architectures for GenAI](#module-04)
5. [Module 05: Multimodal GenAI Pipelines](#module-05)
6. [Module 06: Fine-tuning and Model Customization](#module-06)
7. [Module 07: LLM Output Quality and Reliability](#module-07)

<a id="module-01"></a>
## Module 01: Production-ready API Patterns for GenAI

_Covers robust authentication, error handling, retry logic, fallback strategies, and cost optimization for integrating LLM APIs in production systems._

### LLM API Integration

Production LLM API integration requires robust authentication, error handling, rate limiting, and graceful degradation. Effective patterns include client abstraction, fallback strategies, streaming responses, and cost optimization.

**Key considerations:**

| Concern | Description |
|---|---|
| Authentication security | Secure API key management and token handling |
| Rate limiting | Respect provider quotas; use backoff under throttling |
| Error handling | Classify and recover from transient vs. permanent errors |
| Retry strategies | Automatically retry safe, recoverable failures |
| Timeout management | Enforce strict limits to avoid stalled calls |
| Cost optimization | Cache, batch, and stream to reduce token spend |

### Retry Strategies and Graceful Degradation

| Strategy | What it does |
|---|---|
| Retry strategies | Automatically retry failed operations safely |
| Graceful degradation | Maintain core functionality during failures |

**Retry best practices:**
- Use exponential backoff with jitter
- Retry only transient and recoverable errors
- Enforce retry limits to prevent overload

**Degradation best practices:**
- Serve cached responses during outages
- Degrade to simpler models or responses

### Reliability, Latency, and Cost Trade-offs

The three dimensions of production LLM API performance are in constant tension:

- **Reliability** — stable uptime, error recovery, circuit breakers
- **Reducing latency** — streaming, connection pooling, lower-latency models
- **Cost optimization** — prompt caching, batching, selective streaming

Improving one often pressures the others; explicitly measure and track all three.

### Robust API Client Wrappers

Abstract provider-specific APIs behind a unified client that standardizes authentication, retries, timeouts, streaming, and error handling. This pattern is called the **client abstraction layer** — it lets you swap models or vendors without impacting application logic.

#### Streaming, Timeouts, and Concurrency

| Dimension | Pattern |
|---|---|
| Streaming | Process tokens incrementally to reduce perceived latency |
| Timeouts | Apply strict limits to avoid stalled API calls and blocking resources |
| Concurrency | Limit parallel requests to respect rate limits and maintain system stability |

#### Standardized Interactions with LLM APIs

A unified client interface abstracts provider-specific APIs and normalizes inputs and outputs. It allows models or vendors to be swapped without impacting application logic.

### Fallback Strategies and System Resilience

When the primary model or provider is unavailable, a resilience layer ensures continued service:

- **Service availability** — detect failures before they reach users
- **Model fallback routing** — automatically route to an alternative model
- **Circuit breakers** — stop sending requests when failure rate exceeds threshold
- **Health checks** — continuously probe backend availability
- **Graceful degradation** — serve partial or cached responses rather than hard errors
- **Observability alerts** — surface failures immediately for operator response

**Monitoring for reliability:**
- Detect API failures and latency spikes
- Monitor request volumes
- Alert on error rates and timeouts
- Track fallback and circuit breaker activation
- Validate end-to-end request health

### Optimizing API Usage and Cost

#### Prompt Caching and Request Batching

- **Prompt caching** — reuse identical prompts to avoid repeated API calls for the same input
- **Request batching** — group multiple requests into a single API call to reduce overhead

#### Streaming Optimization

- Streaming returns output incrementally, reducing perceived latency
- Process tokens or chunks as they arrive
- Render text progressively in the UI
- Handle interrupted streams gracefully
- Use streaming selectively — it does not always reduce cost

#### Measuring API Cost Savings

To quantify optimization impact:
- Track token usage per request
- Compare costs before and after optimizations
- Measure savings from prompt caching
- Evaluate cost reductions from request batching

### Demo: Integrating LLM APIs in Production

This demo implements a production-grade LLM client with prompt caching, retry logic, model fallback, and request batching.

In [1]:
import os
import time
import hashlib
import requests
from typing import List

PRIMARY_MODEL = "gpt-4o-mini"
FALLBACK_MODEL = "gpt-4o-mini"

API_URL = "https://api.openai.com/v1/responses"
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")

MAX_RETRIES = 3
TIMEOUT = 15

In [2]:
PROMPT_CACHE = {}

def cache_key(prompt: str, model: str) -> str:
    raw = f"{model}:{prompt}".encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def get_from_cache(prompt: str, model: str):
    return PROMPT_CACHE.get(cache_key(prompt, model))

def save_to_cache(prompt: str, model: str, response: str):
    PROMPT_CACHE[cache_key(prompt, model)] = response


def call_llm_api(prompt: str, model: str) -> str:
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": model,
        "input": prompt
    }
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(API_URL, headers=headers, json=payload, timeout=TIMEOUT)
            resp.raise_for_status()
            return resp.json()["output"][0]["content"][0]["text"]
        except Exception as e:
            print(f"  Attempt {attempt} failed: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)
    raise RuntimeError("All retries exhausted")


def generate(prompt: str) -> str:
    cached = get_from_cache(prompt, PRIMARY_MODEL)
    if cached:
        print("Cache hit")
        return cached

    try:
        print("Using primary model")
        response = call_llm_api(prompt, PRIMARY_MODEL)
        save_to_cache(prompt, PRIMARY_MODEL, response)
        return response
    except Exception as primary_err:
        print(f"Primary failed: {primary_err}. Trying fallback...")
        try:
            response = call_llm_api(prompt, FALLBACK_MODEL)
            return response
        except Exception as fallback_err:
            return f"[Degraded] Could not generate response: {fallback_err}"


def batch_generate(prompts: List[str]) -> List[str]:
    return [generate(prompt) for prompt in prompts]


prompts = [
    "Explain prompt caching in one sentence.",
    "Explain request batching in one sentence.",
    "Explain prompt caching in one sentence."  # intentional duplicate to show cache hit
]

responses = batch_generate(prompts)

for i, r in enumerate(responses, 1):
    print(f"\n--- Response {i} ---")
    print(r)

Using primary model
Using primary model
Cache hit

--- Response 1 ---
Prompt caching is a technique where previously used input prompts and their corresponding outputs are stored to optimize response times and reduce redundant processing in AI systems.

--- Response 2 ---
Request batching is the process of aggregating multiple requests into a single operation to reduce the number of network calls and improve efficiency in communication between a client and server.

--- Response 3 ---
Prompt caching is a technique where previously used input prompts and their corresponding outputs are stored to optimize response times and reduce redundant processing in AI systems.


<a id="module-02"></a>
## Module 02: Structuring and Validating Output Generation

_Covers JSON schema design, Pydantic-based validation, error detection, recovery strategies, and the trade-offs between native structured outputs and freeform text parsing._

### Why Structured Outputs Matter

Structured outputs reduce ambiguity and improve reliability in downstream systems. Validation ensures LLM responses meet expected formats and constraints.

Unstructured LLM output creates integration pain: downstream systems must parse freeform text, which breaks when the model changes its wording, adds narrative prose, or omits fields. Structured outputs shift the contract to the schema level.

**Tools for structuring LLM outputs:**
- **JSON Schema** — define the shape of the expected response
- **Pydantic** — Python-native schema validation with detailed error messages
- **Native structured output APIs** — OpenAI JSON mode, Anthropic tool use
- **Hybrid validation pipelines** — combine multiple approaches for robustness

### Designing JSON Schemas

When defining output fields with JSON Schema or Pydantic:
- Define output fields explicitly with names and types
- Require mandatory fields; leave optional fields with defaults
- Apply value constraints (min/max, enum values, string patterns)
- Validate outputs against the schema before passing to downstream systems

### LLM Output Errors

LLMs often produce output errors such as invalid JSON, missing required fields, or unexpected data types that break downstream systems. Common failure modes:

- **Missing fields** — required keys absent from the response
- **Hallucinated fields** — extra keys not in the schema
- **Formatting errors** — invalid JSON, broken nesting, truncated output

**Error detection strategies:**
- Validate output against predefined schema
- Catch data type mismatches early
- Flag constraint and range violations
- Detect invalid JSON formatting
- Surface likely hallucinated fields

### Recovery and Fallback Approaches

When validation fails:
- Assume LLM outputs can be incomplete or malformed
- Treat validation errors as recoverable events
- Retry with explicit validation feedback (tell the model what was wrong)
- Repair only missing or invalid fields; preserve valid ones
- Use safe defaults for non-critical data

### Native Structured Outputs vs. Parsing Freeform Text

| Native structured outputs | Parsing freeform text |
|---|---|
| Schema enforced | Post hoc parsing |
| Format guaranteed | Format uncertain |
| Fewer hallucinations | Higher hallucinations |
| Easier validation | Complex validation |

#### OpenAI JSON Mode

OpenAI JSON mode forces the model to return a response that strictly conforms to a predefined JSON structure:

```json
{
  "response_format": {
    "type": "json_object",
    "schema": {
      "user_id": { "type": "integer" },
      "risk_level": { "type": "string", "enum": ["low", "medium", "high"] },
      "reason": { "type": "string" }
    },
    "required": ["user_id", "risk_level"]
  }
}
```

#### Anthropic Tool Usage

Enables models to emit structured, validated outputs by calling explicitly defined tools instead of generating freeform text. This approach reduces formatting errors and hallucinations while enabling deterministic integration with functions, APIs, and production workflows.

### Demo: Output Validation Pipeline

This demo shows Pydantic schema validation against LLM output, error detection, and a recovery retry with an explicit correction prompt.

In [3]:
import os
from openai import OpenAI
import json
from pydantic import BaseModel, ValidationError

from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)


class UserProfile(BaseModel):
    name: str
    age: int
    email: str


def validate_output(text):
    try:
        data = json.loads(text)
        return UserProfile(**data)
    except json.JSONDecodeError:
        print("Not valid JSON")
    except ValidationError as e:
        print("Schema mismatch:", e)
    return None


# Intentionally weak prompt that may produce non-JSON output
prompt = "Create a user profile for a fictional person."

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
)
raw_output = response.choices[0].message.content
print("Raw output:\n", raw_output)

result = validate_output(raw_output)

if result is None:
    recovery_prompt = "Return ONLY valid JSON with fields: name (string), age (int), email (string)."
    retry = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": recovery_prompt}],
    )
    fixed_output = retry.choices[0].message.content
    print("\nRecovery output:\n", fixed_output)
    result = validate_output(fixed_output)

print("\nFinal result:", result)

Raw output:
 **User Profile: Emily Carter**

- **Basic Information:**
  - **Name:** Emily Carter
  - **Age:** 29
  - **Gender:** Female
  - **Location:** Portland, Oregon
  - **Occupation:** Graphic Designer

- **Education:**
  - Bachelor of Fine Arts in Graphic Design from the Rhode Island School of Design (RISD) (2016)

- **Interests and Hobbies:**
  - Art and design
  - Photography, especially nature and urban settings
  - Hiking and outdoor activities
  - Traveling to new cities and countries
  - Cooking and trying out new recipes
  - Attending art exhibitions and cultural events

- **Skills:**
  - Proficient in Adobe Creative Suite (Photoshop, Illustrator, InDesign)
  - Strong illustration and typography skills
  - Basic knowledge of web design and HTML/CSS
  - Photography and photo editing
  - Excellent communication and teamwork abilities

- **Personality Traits:**
  - Creative and imaginative
  - Detail-oriented and organized
  - Open-minded and adaptable
  - Passionate about s

<a id="module-03"></a>
## Module 03: Advanced Prompting for Production Systems

_Covers flexible prompt templates, versioning, A/B testing, validation against edge cases and adversarial inputs, and automated validation pipelines._

### Flexible Prompt Templates

Flexible prompt templates are reusable prompt structures with variables that adapt to different inputs, contexts, and tasks. They enable consistent, scalable, and maintainable prompt design for production LLM systems.

**Template components:**
- **Variables** — inject user, task, and context data dynamically
- **Conditions** — conditional blocks adapt instructions to the scenario
- **Dynamic example selection** — choose few-shot examples that match the current context

#### Role of Variables

- Enable dynamic prompt personalization
- Support conditional logic in prompts
- Reduce prompt duplication and errors
- Improve maintainability across prompt versions
- Enable scalable production prompt systems

#### Role of Conditions

- Conditional blocks adapt instructions per scenario
- Dynamic instructions based on context
- Prompt modularity and fallback handling

#### Role of Dynamic Example Selection

- Select representative examples that match the current input
- Context alignment improves model accuracy
- Bias reduction through diverse example pools
- Adaptive learning signals at runtime

### Prompt Versioning and A/B Testing

Prompts are code. They need versioning, change tracking, and performance measurement.

**Tracking prompt versions:**
- Define a consistent prompt versioning scheme (e.g., semantic versioning)
- Store prompts in version-controlled repositories
- Log prompt versions with every invocation
- Track performance metrics per prompt version
- Run A/B tests across prompt variants
- Visualize trends and detect regressions

**Running A/B tests:**
- Define a clear improvement hypothesis before testing
- Create controlled prompt variants
- Randomly route traffic between variants
- Measure output quality and performance metrics
- Promote statistically better prompts

**Analyzing results:**
- Review performance metrics across versions
- Analyze failure and edge case outputs
- Perform qualitative error analysis
- Iterate prompts based on measured results

### Testing and Validating Prompts

Prevent unexpected failures and unsafe outputs by validating prompt behavior across edge cases, diverse inputs, and adversarial scenarios before production deployment. Catch regressions early and ensure consistent performance as prompts evolve.

**Coverage requirements:**
- Define test coverage for normal inputs
- Include edge cases (empty input, very long input, non-English text)
- Simulate adversarial inputs
- Validate diverse contexts
- Automate prompt testing

#### Adversarial Input Handling

- Define common adversarial input patterns
- Detect prompt injection and jailbreak attempts
- Sanitize and normalize user inputs
- Enforce strict prompt and role boundaries
- Log and monitor adversarial behavior

#### Automated Validation Pipelines

Continuously evaluate prompts against regression, edge, and adversarial test suites before deployment.

### Demo: Prompt Templates and Testing

This demo implements a parametrized prompt template, runs it across multiple test cases with varying inputs, and applies simple output validation (word count check).

In [4]:
import os
from openai import OpenAI
from typing import List, Dict
import pandas as pd

from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)

PROMPT_TEMPLATE = """
You are a helpful assistant.

Task:
Summarize the following text in {tone} tone, using at most {max_words} words.

Text:
{text}
"""

def run_prompt(prompt: str, model: str = "gpt-4o-mini") -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    return response.choices[0].message.content.strip()

test_cases: List[Dict] = [
    {
        "name": "Normal input",
        "tone": "professional",
        "max_words": 25,
        "text": "AI models help automate repetitive tasks and improve decision-making."
    },
    {
        "name": "Edge case: very short text",
        "tone": "professional",
        "max_words": 25,
        "text": "AI."
    },
    {
        "name": "Casual tone",
        "tone": "casual",
        "max_words": 15,
        "text": "Large language models are transforming software development by enabling natural language interfaces."
    },
    {
        "name": "Strict word limit",
        "tone": "professional",
        "max_words": 5,
        "text": "This is a long text that should be summarized into very few words for testing constraints."
    },
]

results = []
for case in test_cases:
    prompt = PROMPT_TEMPLATE.format(tone=case["tone"], max_words=case["max_words"], text=case["text"])
    output = run_prompt(prompt)
    results.append({"Test Case": case["name"], "Input Text": case["text"], "LLM Output": output})

df = pd.DataFrame(results)

def word_count(text: str) -> int:
    return len(text.split())

df["Word Count"] = df["LLM Output"].apply(word_count)
df

,Test Case,Input Text,LLM Output,Word Count
0,Normal input,AI models help automate repetitive tasks and i...,AI models enhance efficiency by automating rep...,13
1,Edge case: very short text,AI.,Artificial Intelligence (AI) refers to the sim...,22
2,Casual tone,Large language models are transforming softwar...,Large language models are changing software de...,12
3,Strict word limit,This is a long text that should be summarized ...,Summarize text within constraints.,4


<a id="module-04"></a>
## Module 04: Function Calling Architectures for GenAI

_Covers LLM function calling mechanics, request–response cycles, function schema design, safe execution, permission enforcement, and consistent output formatting._

### What is Function Calling?

LLMs can generate structured requests to invoke external functions, APIs, or tools instead of only producing text. The model selects functions and parameters based on user intent, enabling deterministic actions like database queries, calculations, and API integrations.

#### Request–Response Cycle

1. LLM analyzes the user prompt and decides if a function call is needed
2. LLM generates a structured request using a predefined schema
3. Application executes the function and returns the result
4. LLM uses the output to respond or determine next steps

#### Why Function Calling Matters in Production

- **Reliability and accuracy** — deterministic tool calls replace hallucinated answers
- **Security and control** — permissions enforced before execution
- **Scalability and maintainability** — clean separation between reasoning and action
- **Composability and automation** — chain multiple tool calls for multi-step workflows

### Designing Function Schemas

Well-designed schemas are the foundation of reliable function calling.

**Naming conventions:**
- Use descriptive, domain-aligned function names
- Use explicit, self-documenting parameter names
- Provide clear parameter descriptions in the schema

**Required vs. optional fields:**
- Required fields define the minimum input; missing required fields block execution
- Optional fields add flexibility and should have defaults
- Optional fields support additional context without breaking the call

**Validation rules:**
- Define basic validation rules: required fields, data types, allowed values, length/range constraints
- Reject or auto-correct invalid inputs before function execution

### Safe Function Execution

Use a secure execution layer to validate requests, enforce permissions, safely invoke functions, and return structured results.

**Error handling:**
- Anticipate and classify common failure types
- Validate function calls before execution
- Handle execution errors gracefully and safely
- Return structured errors to the LLM for recovery
- Enforce timeouts, rate limits, and permission checks

**Permission enforcement:**
- Enforce role-based and context-aware permissions before executing any requested function
- Reject or sandbox unauthorized function calls
- Return safe, structured error responses for unauthorized calls

### Formatting Outputs Consistently

Function results returned to the LLM must be predictably structured:

- Predictable structure with a standard response schema
- Machine-readable formats (JSON)
- Explicit success indicators
- Normalized data fields

### Demo: Multi-step Function Calling

This demo implements an agentic loop where the LLM uses weather and activity recommendation tools across multiple steps to answer a user question.

In [5]:
import os
import json
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)


def get_weather(city: str):
    weather_db = {"London": "rainy", "Dubai": "sunny", "New York": "cloudy"}
    if city not in weather_db:
        raise ValueError(f"No weather data for {city}")
    return {"city": city, "condition": weather_db[city]}


def recommend_activity(weather_condition: str):
    recommendations = {
        "rainy": "Visit a museum or indoor gallery",
        "sunny": "Go for a walk or visit a park",
        "cloudy": "Explore local cafes or markets",
    }
    return {"activity": recommendations.get(weather_condition, "Explore the city")}


tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "The city name"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "recommend_activity",
            "description": "Recommend an activity based on weather condition",
            "parameters": {
                "type": "object",
                "properties": {"weather_condition": {"type": "string", "description": "Weather condition string"}},
                "required": ["weather_condition"],
            },
        },
    },
]


def execute_tool(tool_call):
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    try:
        if name == "get_weather":
            return get_weather(**args)
        elif name == "recommend_activity":
            return recommend_activity(**args)
        else:
            return {"error": f"Unknown function: {name}"}
    except Exception as e:
        return {"error": str(e)}


messages = [{"role": "user", "content": "I am in London today. What should I do?"}]

for step in range(5):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )
    assistant_message = response.choices[0].message

    if not assistant_message.tool_calls:
        print("Final answer:", assistant_message.content)
        break

    messages.append(assistant_message)

    for tool_call in assistant_message.tool_calls:
        result = execute_tool(tool_call)
        print(f"Tool '{tool_call.function.name}' returned: {result}")
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result),
        })

Tool 'get_weather' returned: {'city': 'London', 'condition': 'rainy'}
Tool 'recommend_activity' returned: {'activity': 'Visit a museum or indoor gallery'}
Final answer: The weather in London today is rainy. A good activity to do would be to visit a museum or an indoor gallery. Would you like recommendations for specific museums or galleries?


<a id="module-05"></a>
## Module 05: Multimodal GenAI Pipelines

_Covers multimodal system design, structured extraction from images and documents, API optimization strategies, fallback and resilience patterns for multimodal inputs._

### Introduction to Multimodal Systems

Multimodal systems combine multiple input types — text, images, documents, audio, video — to enable richer GenAI interactions. They allow models to reason across modalities, extracting structured and unstructured information for more robust downstream tasks.

**Key characteristics:**

| Characteristic | Description |
|---|---|
| Multimodal inputs | Accept text, image, document, audio, video |
| Shared embeddings | Represent different modalities in a common space |
| Modular design | Pipeline stages are swappable per modality |
| Data robustness | Handle format variation and missing modalities |
| Scalable pipelines | Process large document batches efficiently |
| Cross-modal reasoning | Draw inferences across modality boundaries |

**Challenges:**

| Challenge | Description |
|---|---|
| Format | Handling incompatible and unsupported input formats |
| Cost | Balancing trade-offs between multimodal accuracy and API cost |
| Processing | Managing complex multimodal pipelines especially at scale |

### Structured Extraction from Multimodal Inputs

Structured extraction is critical for production automation:
- Free-form multimodal outputs are difficult to validate and integrate
- Predictable schemas are required for downstream pipelines
- Structured data improves monitoring, error handling, and retries
- Compliance and governance rely on clearly defined data fields

### Optimizing Multimodal API Usage

#### Image Resolution vs. Cost

- Higher resolution increases accuracy and cost
- Lower resolution reduces cost but lowers quality
- Resize images based on task requirements
- Crop to relevant regions only

#### Token Usage

Token usage directly impacts latency, cost, and scalability. Image resolution, patching strategy, and batching choices all tie directly to overall model token consumption.

#### Batching Strategies

Batch pages and documents by similar size and modality to reduce per-request overhead. Use asynchronous batching with token-aware limits to balance latency, cost, and memory.

| Strategy | Description |
|---|---|
| Size-based | Group by document size |
| Modality-based | Group by input type |
| Token-aware | Respect token limits per request |
| Asynchronous | Non-blocking batch processing |
| Pipeline-stage | Batch at each stage independently |

**Best practices:**
- Design requests with token and rate limit awareness
- Combine batching, caching, and controlled retries for efficient throughput

### Fallback and Resilience Strategies

#### Handling Multimodal Failures

- Detect unsupported formats early
- Convert formats to supported alternatives
- Validate image size and encoding before sending
- Monitor vision confidence signals
- Retry with higher resolution images
- Flag degraded vision processing paths

#### OCR and Text-only Fallback Pipelines

When vision models fail or return low-confidence results:
- Include OCR as a fallback for scanned or low-quality inputs
- Apply text-only processing when vision fails
- Trigger fallbacks on errors or low confidence scores
- This improves system robustness without sacrificing accuracy on good inputs

#### Monitoring for Failures and Graceful Degradation

- **Detection** — identify failures early in the pipeline
- **Observability** — track confidence scores and error rates
- **Graceful Degradation** — maintain partial functionality when full vision fails

### Demo: Multimodal Pipeline with OCR Fallback

This demo processes images and PDFs through a vision model first, then falls back to Tesseract OCR if the vision output is insufficient.

In [6]:
import os
import base64
import io
import fitz  # PyMuPDF
from PIL import Image, ImageDraw, ImageFont
import pytesseract
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)


def image_to_base64(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


def vision_extract_text(image: Image.Image) -> str:
    try:
        response = client.responses.create(
            model="gpt-4.1-mini",
            input=[{
                "role": "user",
                "content": [
                    {"type": "input_text", "text": "Extract all readable text."},
                    {"type": "input_image", "image_url": f"data:image/png;base64,{image_to_base64(image)}"},
                ],
            }],
        )
        return response.output[0].content[0].text
    except Exception:
        return ""


def ocr_extract_text(image: Image.Image) -> str:
    return pytesseract.image_to_string(image)


def bad_text(text: str) -> bool:
    return (not text) or len(text) < 30


def process_image(image: Image.Image) -> str:
    text = vision_extract_text(image)
    if bad_text(text):
        print("  Falling back to OCR")
        text = ocr_extract_text(image)
    return text


def process_pdf(path: str) -> str:
    doc = fitz.open(path)
    pages = []
    for page in doc:
        pix = page.get_pixmap(dpi=200)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        pages.append(process_image(img))
    return "\n".join(pages)


def process_files(files: list) -> dict:
    results = {}
    for f in files:
        print(f"Processing: {f}")
        if isinstance(f, Image.Image):
            results["<in-memory image>"] = process_image(f)
        elif f.lower().endswith((".png", ".jpg", ".jpeg")):
            img = Image.open(f)
            results[f] = process_image(img)
        elif f.lower().endswith(".pdf"):
            results[f] = process_pdf(f)
        else:
            results[f] = "[Unsupported format]"
    return results


# --- Generate sample inputs so the demo runs without external files ---

def make_sample_image() -> Image.Image:
    img = Image.new("RGB", (400, 100), color="white")
    draw = ImageDraw.Draw(img)
    draw.text((10, 30), "Invoice #1234  Amount: $99.00  Due: 2024-12-01", fill="black")
    return img


def make_sample_pdf_bytes() -> bytes:
    """Create a minimal single-page PDF with text using PyMuPDF."""
    doc = fitz.open()
    page = doc.new_page(width=400, height=100)
    page.insert_text((10, 60), "Order #5678  Product: Widget  Qty: 3", fontsize=14)
    buf = io.BytesIO()
    doc.save(buf)
    return buf.getvalue()


sample_img = make_sample_image()
sample_pdf_path = "/tmp/sample_demo.pdf"
with open(sample_pdf_path, "wb") as f:
    f.write(make_sample_pdf_bytes())

# Process in-memory image directly, and the generated PDF
print("--- Sample image ---")
print(process_image(sample_img))

print("\n--- Sample PDF ---")
print(process_pdf(sample_pdf_path))


--- Sample image ---
The readable text in the image is:

"Invoice #1234 Amount $99.00 Due: 2024-12-01"

--- Sample PDF ---
The readable text in the image is:

"Order #5678 Product: Widget Qty: 3"


<a id="module-06"></a>
## Module 06: Fine-tuning and Model Customization

_Covers when to use fine-tuning vs. prompt engineering vs. RAG, fine-tuning pipelines, parameter-efficient techniques (LoRA/QLoRA), and cost optimization._

### Choosing the Right Approach: Fine-tuning, Prompt Engineering, RAG

**Fine-tuning** — use when:
- Pre-trained models fail to capture domain-specific visual patterns, terminology, or document layouts
- You have sufficient high-quality labeled data
- Input distributions are stable and justify the added cost and operational complexity

**Prompt engineering** — use when:
- Coordinating multiple models (vision, OCR, text) to control how outputs are combined
- Converting model outputs into structured results
- Quick iteration is needed without retraining

**Retrieval Augmented Generation (RAG)**:
- Combines information retrieval with an LLM
- Grounds responses using external knowledge
- Uses embeddings and vector databases for retrieval
- Injects retrieved context into the prompt
- Reduces hallucinations and improves factual accuracy

### Fine-tuning Pipelines

#### Preparing Datasets

1. Define target task and expected model behavior
2. Normalize input and output formats
3. Clean and filter data to remove noise
4. Split data into training, validation, and test sets
5. Validate dataset size and class balance

#### Running and Monitoring Training Jobs

1. Configure training parameters (learning rate, epochs, batch size)
2. Launch fine-tuning jobs on the GenAI platform
3. Monitor training metrics such as loss during execution
4. Track resource usage to manage cost and duration
5. Capture logs and checkpoints for debugging and recovery

#### Managing Model Versions

- Assign clear version identifiers to each fine-tuned model
- Store metadata: dataset version, parameters, training date
- Compare model performance across versions using benchmarks
- Promote validated models to staging or production
- Retire outdated models to reduce maintenance and operational risk

### Parameter-efficient Fine-tuning (PEFT)

Full fine-tuning requires updating all model parameters — prohibitively expensive for large models. PEFT techniques reduce cost dramatically.

#### LoRA and QLoRA

| Technique | Description |
|---|---|
| LoRA (Low-Rank Adaptation) | Adds small trainable low-rank matrices to frozen base model weights |
| QLoRA | LoRA applied to a quantized (4-bit) base model — extreme memory efficiency |
| Parameter-efficient fine-tuning | Only adapter weights are updated; base model stays frozen |
| Limited infrastructure | Can fine-tune on a single GPU rather than a cluster |
| Domain-specific adaptation | Task-specific behavior without full retraining |

#### Switching Between Multiple LoRA Adapters

Adapters enable multiple task-specific behaviors on a single base model. Runtime adapter switching supports:
- Experimentation and A/B testing of different domain adapters
- Multi-tenant deployments where different users need different behaviors

### Optimizing Compute and Storage Costs

| Technique | Benefit |
|---|---|
| Image downscaling | Reduce vision token cost |
| Token optimization | Minimize prompt and context length |
| Request batching | Amortize per-request overhead |
| Text-only fallback | Avoid vision model cost when unnecessary |
| Document compression | Reduce input size for large documents |
| Result caching | Avoid redundant inference for repeated inputs |

### Demo: Evaluating Fine-tuned Models

This demo compares a base model, a prompt-engineered model, and a simulated fine-tuned model on the same task, measuring latency, token usage, and a simple quality score.

In [7]:
import os
import time
import pandas as pd
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)

TEST_PROMPT = """
Summarize the following customer complaint in one sentence:

"I was charged twice for my subscription this month and customer support has not responded to my emails."
"""

MODELS = [
    {
        "name": "Base Model",
        "model_id": "gpt-4o-mini",
    },
    {
        "name": "Prompt-Engineered Model",
        "model_id": "gpt-4o-mini",
        "system_prompt": "You are an expert customer support analyst. Be concise and accurate.",
    },
    {
        "name": "High-Quality Model (Fine-Tune Equivalent)",
        "model_id": "gpt-4o-mini",
        "system_prompt": "You are a highly specialized customer complaint analyst. Extract the core issue in exactly one sentence.",
    },
]


def run_model(model_config, prompt):
    start_time = time.time()
    messages = []
    if "system_prompt" in model_config:
        messages.append({"role": "system", "content": model_config["system_prompt"]})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=model_config["model_id"],
        messages=messages,
    )
    latency = (time.time() - start_time) * 1000
    output = response.choices[0].message.content
    tokens = response.usage.total_tokens
    return output, latency, tokens


results = []
for m in MODELS:
    output, latency, tokens = run_model(m, TEST_PROMPT)
    results.append({
        "model": m["name"],
        "output": output,
        "latency_ms": round(latency, 1),
        "tokens_used": tokens,
    })

df = pd.DataFrame(results)


def quality_score(text):
    keywords = ["charged", "twice", "support"]
    score = sum(k in text.lower() for k in keywords)
    return score / len(keywords)


df["quality_score"] = df["output"].apply(quality_score)

COST_PER_1K_TOKENS = {
    "Base Model": 0.002,
    "Prompt-Engineered Model": 0.003,
    "High-Quality Model (Fine-Tune Equivalent)": 0.0015,
}
df["cost_usd"] = df.apply(
    lambda row: (row["tokens_used"] / 1000) * COST_PER_1K_TOKENS[row["model"]], axis=1
)

df[["model", "output", "latency_ms", "tokens_used", "quality_score", "cost_usd"]]

,model,output,latency_ms,tokens_used,quality_score,cost_usd
0,Base Model,The customer is frustrated after being charged...,988.2,57,1.000000,0.000114
1,Prompt-Engineered Model,The customer is frustrated about being charged...,1022.9,75,1.000000,0.000225
2,High-Quality Model (Fine-Tune Equivalent),The customer is frustrated over being double c...,921.9,79,0.666667,0.000119


<a id="module-07"></a>
## Module 07: LLM Output Quality and Reliability

_Covers output validation pipelines, confidence estimation, multi-level validation, output refinement, and standardizing outputs across systems._

### Validating LLM Outputs

LLM outputs can be fluent but unreliable, making validation essential to detect hallucinations and inconsistencies. Output validation pipelines ensure trust, safety, and production readiness by combining automated checks, confidence estimation, and continuous feedback.

**Practical validation strategies:**
- **Fact-checking** — verify claims against source data
- **Consistency** — compare output with prior outputs for the same input
- **Safety filters** — detect harmful, biased, or off-topic content
- **Business rules** — enforce domain-specific constraints

#### Identifying Common LLM Output Errors

| Error type | Description |
|---|---|
| Hallucinations | Confident but incorrect facts not grounded in source data |
| Formatting issues | Responses that violate expected structure, schema, or layout |

#### Why Multi-level Validation Is Critical

- LLMs can fail in different ways simultaneously
- No single validation technique is sufficient
- Risk increases with autonomy and integration depth
- Different failure modes require different controls
- Layered validation improves trust and scalability

### Confidence and Uncertainty

**Confidence** measures how strongly the model supports its output. **Uncertainty** flags when the model is unsure or unstable.

#### Measuring LLM Confidence

| Method | How it works |
|---|---|
| Logprobs | Token-level probabilities estimate output likelihood |
| Multiple samples | Compare repeated outputs to assess stability and agreement |

#### Responding to Uncertain LLM Results

- Detect low-confidence or high-variance model outputs
- Route flagged responses for human review or approval
- Automatically rerun generation with adjusted prompts or parameters
- Apply fallback logic or alternative models when uncertainty remains
- Log uncertainty signals to improve monitoring and system tuning

**Using confidence metrics for decisions:**
- Apply thresholds to accept or review outputs
- Route low-confidence results to fallbacks
- Gate actions using confidence levels
- Log scores for monitoring and continuous improvement

### Output Refinement

Raw LLM outputs often need post-processing before they are production-ready:

| Refinement type | What it does |
|---|---|
| Rewriting | Improve clarity, structure, or tone |
| Enhancement | Add missing context or detail |
| Summarization | Reduce length while preserving meaning |

**Post-processing and rule-based fixes:**
- **Output normalization** — standardize formats, casing, whitespace
- **Rule enforcement** — apply business rules and domain constraints
- **Safety guardrails** — filter unsafe or off-topic content
- **Domain augmentation** — enrich with domain-specific knowledge
- **Explainable fixes** — record what was changed and why

### Standardizing Outputs Across Systems

Consistent output schemas enable reliable downstream integration across systems and teams, and simplify multi-LLM interoperability.

**Key standardization strategies:**
- Use canonical output schemas
- Enforce controlled vocabularies and enums
- Normalize dates, units, confidence scores, and identifiers
- Separate content, metadata, and quality signals

### Demo: Feedback Loops for Output Quality

This demo simulates a feedback loop that collects user ratings and automated quality checks over multiple iterations, then analyzes reliability metrics across prompt versions.

In [8]:
import random
import pandas as pd


def generate_output(prompt, prompt_version="v1"):
    if prompt_version == "v1":
        return f"Answer (v1): This is a generic response to '{prompt}'."
    elif prompt_version == "v2":
        return f"Answer (v2): This is a clearer, more specific response addressing '{prompt}'."
    else:
        return f"Answer ({prompt_version}): Response to '{prompt}'."


def collect_user_feedback():
    return random.randint(1, 5)


def automated_quality_check(output):
    return len(output) > 50


feedback_log = []
prompt = "Explain feedback loops in AI systems"
prompt_version = "v1"

for iteration in range(5):
    output = generate_output(prompt, prompt_version)
    user_rating = collect_user_feedback()
    auto_check_passed = automated_quality_check(output)

    feedback_log.append({
        "iteration": iteration + 1,
        "prompt_version": prompt_version,
        "output": output,
        "user_rating": user_rating,
        "auto_check": auto_check_passed,
    })

    # Upgrade to v2 if rating drops below threshold
    if user_rating < 3:
        prompt_version = "v2"

df = pd.DataFrame(feedback_log)
print(df.to_string(index=False))

metrics = {
    "average_user_rating": df["user_rating"].mean(),
    "auto_check_pass_rate": df["auto_check"].mean(),
}
pd.DataFrame([metrics])

 iteration prompt_version                                                                                                    output  user_rating  auto_check
         1             v1                        Answer (v1): This is a generic response to 'Explain feedback loops in AI systems'.            2        True
         2             v2 Answer (v2): This is a clearer, more specific response addressing 'Explain feedback loops in AI systems'.            1        True
         3             v2 Answer (v2): This is a clearer, more specific response addressing 'Explain feedback loops in AI systems'.            2        True
         4             v2 Answer (v2): This is a clearer, more specific response addressing 'Explain feedback loops in AI systems'.            2        True
         5             v2 Answer (v2): This is a clearer, more specific response addressing 'Explain feedback loops in AI systems'.            5        True


,average_user_rating,auto_check_pass_rate
0,2.4,1.0
